# 第03篇：C语言程序控制计算机系统硬件运行

> **工程与学习规范约束（必读）**
> 
> 本教程采用“边阅读原理，边生成代码，边上板运行”的沉浸式交互结构。
> 1. **代码目录统一放**：`/home/frank/embedded/examples/ch03_imx8m_overview/` 及其子目录中。
> 2. **产物输出分离**：可执行文件输出到 `bin/`；中间对象文件输出到 `build/`。
> 3. **构建自动化**：所有包含 C 程序的目录，均配套 `Makefile`。
> 4. **NFS 联调机制**：
>    - 主机（PC）端工作区：`~/nfs_share`
>    - 开发板挂载点：`/mnt/nfs`
>    - 默认开发板 IP：`192.168.137.2`


## 第1章：从软件到硬件：CPU 与外设的统一控制

在《第02章：计算机系统硬件组成》中，我们已经明白了：**ALU（算术逻辑单元）执行运算时，需要控制器提供精确的“控制信号”**。而这些控制信号，正是由**指令译码**产生的。所谓的“指令”，不过是程序员想要的运算与运算数的二进制形式，如何产生这些指令呢，就是按照指令集，通过编译器编译或者解释器解释得到的。C程序要能在CPU上执行，就必须按照CPU的指令集进行编译，然后将其放置在RAM上，上电后就可以按照上一章讲解的流程：取指-译码-执行了。本章讲述如何编写C程序并且利用特定指令集的计算机系统的CPU、RAM运行，并最终控制外部引脚设备。


### 1.1 C 语言程序的编译流程
#### 1.1.1 实例：C 程序的完整编译流程 (Compilation Process)
你在 PC 上敲下的 C 代码，必须经过交叉编译器 (Cross-compiler) 的四步流水线，才能变成开发板能执行的纯粹机器码。这四步分别是：
1. **预处理 (Preprocessing)**：展开 `#include` 和 `#define`，去除注释。产物：`.i` 文件。
2. **编译 (Compilation)**：将 C 语言翻译为特定 CPU 架构（如 aarch64, x86_64 等）的汇编语言。**CPU 架构的核心标志就是其指令集架构 (ISA)**，不同的指令集对应完全不同的汇编语言和机器码。这就是为什么我们在《实验二》中必须使用**交叉编译器**（如 `aarch64-linux-gnu-gcc`），以此来在 x86_64 的 PC 上，翻译出目标板 (aarch64) 能听懂的专属汇编语言。产物：`.s` 文件。
3. **汇编 (Assembly)**：将汇编语言翻译为二进制机器指令 (Machine Code)。产物：`.o` 目标文件。
4. **链接 (Linking)**：将多个 `.o` 文件和标准库连接在一起，并进行**内存分段 (Memory Segmentation)**。最终被切分为 `.text`（只读机器码指令）、`.data`（有初值的全局变量）、`.bss`（无初值变量）等物理段。断电状态下，它们安静地躺在 ROM (如 eMMC/SD卡) 中。


💡 **温馨提示**：GCC讲解及使用可参考**教材**：P76-P77 3.1.1-3.1.2图3.1之前内容

为了直观地看到编译流程和内存分段，我们来编写一个包含多种变量的 `hello.c`，并使用 `gcc` 的不同参数和 `size` 命令来一探究竟。


In [ ]:
%%writefile /home/frank/embedded/examples/ch03_imx8m_overview/hello.c
#include <stdio.h>

#define MESSAGE "Hello i.MX8MM!"

// 存放在 .data 段 (已初始化的全局变量)
int global_init_var = 42;

// 存放在 .bss 段 (未初始化的全局变量，在文件中不占实际体积，上电清零)
int global_uninit_var;

int main() {
    // 存放在栈 (Stack) 中
    int local_var = 10;
    
    // 存放在 .text 段 (只读机器指令)
    printf("%s\n", MESSAGE);
    printf("Init var: %d, Uninit var: %d, Local var: %d\n", 
           global_init_var, global_uninit_var, local_var);
           
    return 0;
}


💡 **温馨提示**：如下编译命令及编译选项见**教材**P77-P83 3.1.2图3.1之后内容-3.1.3

mkdir命令见**教材** P222 表A.1及P223 3.mkdir命令

file命令见**教材** P230 表A.3及P232 5.file

建议掌握**教材** 附录A 常用Linux命令的使用

常用Linux命令可利用utools中的Linux命令文档插件进行查询

In [ ]:
# 步骤 0：创建标准的中间产物与可执行文件目录
!mkdir -p /home/frank/embedded/examples/ch03_imx8m_overview/build
!mkdir -p /home/frank/embedded/examples/ch03_imx8m_overview/bin

# 步骤 1：预处理 (-E)，查看宏定义是否被替换。中间产物放入 build/ 目录
!gcc -E /home/frank/embedded/examples/ch03_imx8m_overview/hello.c -o /home/frank/embedded/examples/ch03_imx8m_overview/build/hello.i
# 💡 使用 tail -n 15 命令：查看 hello.i 文件的最后 15 行，可以观察到最底部的 main 函数以及被替换后的宏定义
!tail -n 15 /home/frank/embedded/examples/ch03_imx8m_overview/build/hello.i


In [ ]:
# 步骤 2：编译 (-S)，生成汇编代码。中间产物放入 build/ 目录
!gcc -S /home/frank/embedded/examples/ch03_imx8m_overview/build/hello.i -o /home/frank/embedded/examples/ch03_imx8m_overview/build/hello.s
# 💡 使用 head -n 20 命令：查看 hello.s 文件的开头 20 行，可以直观地看到生成的汇编语言指令和段声明（如 .text, .data）
!head -n 20 /home/frank/embedded/examples/ch03_imx8m_overview/build/hello.s


In [9]:
# 步骤 3：汇编 (-c)，生成目标文件。中间产物放入 build/ 目录
!gcc -c /home/frank/embedded/examples/ch03_imx8m_overview/build/hello.s -o /home/frank/embedded/examples/ch03_imx8m_overview/build/hello.o
# 💡 使用 file 命令：探测 hello.o 文件的具体类型和架构信息，确认它是否已经变成了无法直接阅读的二进制目标文件
!file /home/frank/embedded/examples/ch03_imx8m_overview/build/hello.o

# 步骤 4：链接，生成可执行文件放入 bin/ 目录，并使用 size 命令查看内存分段 (Memory Segmentation)
!gcc /home/frank/embedded/examples/ch03_imx8m_overview/build/hello.o -o /home/frank/embedded/examples/ch03_imx8m_overview/bin/hello_x86
!size /home/frank/embedded/examples/ch03_imx8m_overview/bin/hello_x86



/home/frank/embedded/examples/ch03_imx8m_overview/build/hello.o: ELF 64-bit LSB relocatable, x86-64, version 1 (SYSV), not stripped
   text	   data	    bss	    dec	    hex	filename
   1564	    612	     12	   2188	    88c	/home/frank/embedded/examples/ch03_imx8m_overview/bin/hello_x86


#### 1.1.2 编译后的C语言程序如何加载到CPU上运行
经过上述 4 步生成的二进制机器码，目前只是安静地躺在 PC 硬盘里，你需要把它传输到开发板的非易失性存储器中（如 NOR Flash, NAND Flash, eMMC, SD卡等）。

**上电后，它的硬件执行流程如下：**
1. **上电加载 (Loading into RAM)**：通电瞬间，处理器内部固化的极其底层的引导代码（BootROM）会根据特定的启动配置，去初始化外部的非易失性存储介质（即广义上的 **ROM**），并将存放在里面的程序（Bootloader/系统/应用）批量搬运到速度极快的 DDR 内存（**RAM**）中。只有进入 RAM 后，程序才成为 CPU 可以直接取用执行的“活跃舞台”。
   
   > **💡 结合开发板的实际例子：eMMC 启动机制**
   > 在我们配套的开发板上，如果你将拨码开关（BOOT Switch）拨到 `0, 0`，就是告诉主处理器采用 **eMMC 启动**。
   > - **eMMC (Embedded Multi-Media Controller, 嵌入式多媒体存储卡)**：本质上是一块集成了高密度 NAND Flash 闪存和主控芯片的存储介质，相当于手机和嵌入式板卡里的“小硬盘”。
   > - 当拨码为 `0, 0` 时，上电后 i.MX8MM 内部的 BootROM 会立刻检测到这个电平信号，主动去初始化 eMMC，并从中读取 Linux 引导程序加载到 DDR (RAM) 里。

2. **取指 (Fetch)**：RAM 中指令的物理地址被逐个传给 PC (Program Counter)。控制器通过 PC 从 RAM 读取 32 位机器指令，暂存在 **IR (Instruction Register)** 中。
3. **译码 (Decode)**：控制器内部的译码器将 IR 中的二进制串“翻译”为高低电平控制信号 (Control Signals)。
4. **执行 (Execute)**：ALU (Arithmetic Logic Unit) 接收控制信号完成运算，或数据通路完成 RAM 到寄存器的读写。


### 1.2 C程序如何控制外设

在《第02章：计算机系统硬件组成》中，我们了解了低速外设是通过 IO 接口电路与 CPU 相连的。那么，C 程序控制外设的科学思路是什么呢？

#### 1.2.1 C 程序控制外设的通用思路（五步法）
要控制任何一个外设，无论是简单的 LED 还是复杂的传感器，标准的流程都可以归纳为以下五个步骤：

1. **查看要控制设备的工作原理**：例如，控制 LED，我们需要知道它是共阳极还是共阴极，高电平点亮还是低电平点亮。
2. **查看它是如何与 IO 口相连的**：通过查阅**开发板原理图**，找到外设连接到了主芯片的哪一个具体引脚（如 `GPIO1_IO12`）。
3. **查看通过 MMIO 分配的工作模式寄存器和数据寄存器的地址**：通过查阅**芯片参考手册 (Reference Manual)**，找到该 IO 口对应控制器的物理基地址，以及其内部的**工作模式寄存器**（如方向寄存器 GDIR）和**数据寄存器**（如数据寄存器 DR）的具体地址偏移。
4. **配置工作模式寄存器**：往模式寄存器中写入特定的控制字，确定该引脚是输入还是输出、是模拟输入还是数字输入。
5. **根据工作模式读取、写入数据寄存器，完成控制**：如果是输出模式，往数据寄存器写入 1 或 0 即可改变引脚电平；如果是输入模式，从数据寄存器读取 1 或 0 即可获取外部状态。


#### 1.2.2 嵌入式 C 语言程序控制外部设备设计规则

相较于在 PC 上跑的普通 C 程序（只关心算法和内存里的变量），嵌入式 C 程序控制外设新增了寄存器变量定义和寄存器变量赋值两个操作。为此，我们给出嵌入式C语言程序控制外部设备程序设计的三大基本规则：

##### 规则 1：定义寄存器变量的完整推导过程与 `volatile` 防优化
在 C 语言中，我们要操作一个物理地址（比如外设寄存器地址 `0x30200000`），不能直接把数字赋值给硬件。这是一个循序渐进的推导过程：

**第一步：将十六进制数值转换为地址（指针强转）**
在编译器眼中，`0x30200000` 只是一个普通的整数。我们需要通过**强制类型转换**，告诉编译器：“这是一个内存地址，且该地址存放的是一个 32 位（4字节）的数据”。
```c
// 此时，0x30200000 变成了一个指向 32 位无符号整数的指针
(uint32_t *)0x30200000;
```

**第二步：解引用（读写该地址的内容）**
有了指针后，我们在前面加上解引用符号 `*`，就可以向这个物理地址写数据了：
```c
// 向该寄存器地址写入数值 1
*((uint32_t *)0x30200000) = 1; 
```

**第三步：引入 `volatile` 防止编译器“自作聪明”**
**这是嵌入式开发最致命的陷阱**。假设我们写了这样一段代码来等待按键按下（按键按下时硬件会将该地址的值变为 1）：
```c
// ❌ 灾难性的错误写法
uint32_t *btn_status = (uint32_t *)0x30200000;
while (*btn_status == 0) { 
    // 死循环等待按键按下
}
```
现代编译器（如 GCC 加了 `-O2` 优化）非常聪明，它会认为：“你在循环里没有修改 `btn_status`，那这个物理内存的值肯定不会变，为了提高性能，我把这个值读取一次，缓存到 CPU 内部的超高速寄存器（如 `X0`）里，以后每次循环只检查 `X0` 寄存器就好了。”

**后果就是：** 无论你在外部怎么狂按按键，物理内存 `0x30200000` 里的值虽然变成了 1，但 CPU 一直在死抠它的内部缓存 `X0`（值永远是 0），导致程序永远卡死在循环里！

因此，我们必须加入 `volatile`（易变的、挥发性的）关键字修饰指针，警告编译器：
> **“这个地址的数据随时会被外部硬件改变！绝不允许做任何缓存优化，每一次执行 `*` 解引用，你都必须老老实实地走系统总线，去物理地址把最新数据取回来！”**

**最终的标准形态：**
```c
// ✅ 标准的单寄存器定义宏（内核源码中极其常见）：
#define GPIO1_DR  (*((volatile uint32_t *)0x30200000))

// 这样我们就可以直接像操作普通变量一样操作硬件了：
GPIO1_DR = 1;  // 写入数据
```

##### 规则 2：使用标准固定宽度类型与结构体 (Struct) 映射
**原理（为什么要引入规则 2？）**：
你可能会问：“规则 1 已经能定义寄存器了，规则 2 会不会重复？”
答案是：**规则 1 适用于只操作一两个孤立寄存器的简单场景。** 但在真实的嵌入式芯片中，一个外设（如 GPIO1 模块）往往拥有几十甚至上百个控制不同特性的寄存器，它们在物理内存地址上是**连续递增**的。如果用规则 1 去定义，你得手写几百行容易算错地址的宏定义。

**规范做法**：
**规范**：
1. 绝对不要用 `int` 或 `long`，必须包含 `<stdint.h>` 并使用 `uint32_t`。因为不同的 CPU 架构下 `int` 长度可能不同，而硬件寄存器是严格的 32 位（4字节）。
2. 不要手动计算偏移量（如 `base + 4`, `base + 8`），应利用 C 语言结构体成员在内存中连续排列的特性，让编译器自动算偏移。

```c
#include <stdint.h>

typedef struct {
    volatile uint32_t DR;    // 偏移 0x00：数据寄存器
    volatile uint32_t GDIR;  // 偏移 0x04：方向寄存器
    volatile uint32_t PSR;   // 偏移 0x08：状态寄存器
} GPIO_Regs;

// 将物理基地址强转为结构体指针
GPIO_Regs *gpio1 = (GPIO_Regs *)0x30200000;
// 此时 gpio1->GDIR 就完美映射到了 0x30200004
```

> **💡 深度解析：初学者如何理解“将物理基地址强转为结构体指针”？**
> 
> 很多初学者看到 `(GPIO_Regs *)0x30200000` 会感到困惑。你可以用**“透明的刻度尺（模板）”**来形象地理解这个过程：
> 
> 1. **结构体只是一个“透明模板”**：当你写下 `typedef struct { ... } GPIO_Regs;` 时，内存里并没有真正分配空间。你只是告诉编译器制作了一把“刻度尺”：第一格叫 `DR`（宽4字节），第二格叫 `GDIR`（宽4字节），第三格叫 `PSR`（宽4字节）。
> 2. **基地址是“起点”**：`0x30200000` 是硬件手册上规定的 GPIO1 模块在物理内存中的绝对起点位置。
> 3. **强转就是“把模板严丝合缝地盖在内存上”**：强制类型转换 `(GPIO_Regs *)0x30200000` 的本质动作，就是把这把“透明刻度尺”的 `0` 刻度，精准对齐到物理内存的 `0x30200000` 处。
> 
> **编译器的魔法**：盖上去之后，当你写出 `gpio1->GDIR` 时，编译器查刻度尺发现 `GDIR` 在第二格（偏移 4 个字节），于是它在底层自动帮你翻译成了对物理地址 `0x30200000 + 0x04 = 0x30200004` 的直接访问。这完美消除了程序员手动计算 `0x30200004`, `0x30200008` 导致算错地址的致命风险！

##### 规则 3：使用“读-改-写 (Read-Modify-Write)”进行位操作
**原理**：一个 32 位寄存器往往控制着 32 个不同的引脚。如果我们只想点亮第 12 号引脚的 LED，绝不能直接写 `gpio1->DR = 1;`（这叫“盲写”，会把其他 31 个引脚的状态全部破坏）。
**规范**：必须使用位操作符 `&`（与）、`|`（或）、`~`（非）。

*   **置 1（设为高电平）**：使用 `|=`
    *   `gpio1->DR |= (1 << 12);` （仅把第 12 位置 1，其余位不变）
*   **清 0（设为低电平）**：使用 `&= ~`
    *   `gpio1->DR &= ~(1 << 12);` （仅把第 12 位清 0，其余位不变）
*   **翻转（高变低，低变高）**：使用 `^=`
    *   `gpio1->DR ^= (1 << 12);` 


### 1.3 什么是“裸机 (Bare-metal)”开发？

在进入下一节的具体代码之前，我们需要先明确一个极其重要的概念：**裸机开发**。

> **裸机 (Bare-metal)** 是指在**完全没有操作系统 (OS, 如 Linux、RTOS) 支持**的环境下，直接让我们的应用程序在物理硬件上运行的开发模式。

在裸机环境中：
1. **独占 CPU 执行权**：程序通常是一个死循环（Super-loop），独占 CPU 的所有执行时间，没有多任务调度，也没有后台进程。
2. **直接操作物理内存**：程序直接读写真实的物理地址（例如直接向 `0x30200000` 写入数据），没有虚拟内存（MMU）的隔离与保护。
3. **亲手控制底层硬件**：开发者需要亲自查阅芯片手册，通过指针强制转换，直接操控底层的硬件寄存器。

本章我们将要编写的 LED 闪烁程序，就是典型的“裸机程序”。这种开发模式能让我们深刻理解软硬件交互的最底层本质。但随着业务逻辑变得复杂（例如需要同时处理网络通信、屏幕显示、多任务并发时），裸机开发会遇到灾难性的资源竞争和代码维护危机。**这也正是为什么在下一章《Linux系统原理与驱动模型架构》中，我们必须从裸机演进到操作系统的原因。**

### 1.4 实例：利用裸机 C 程序控制 i.MX8MM LED（基于 Renode 仿真）

真实的 i.MX8MM 开发板通常固化了 U-Boot 和 Linux 系统，上电后 Linux 会接管所有硬件。如果我们强行在 A53 核心上运行裸机程序，需要编写极其复杂的时钟、DDR 和安全环境初始化代码。
为了专注于“**C语言如何控制外设**”这一核心逻辑，我们将使用 **Renode 仿真器** 来模拟一个干净的 i.MX8MM 裸机环境，并运行一段标准的嵌入式 C 代码。

#### 1.4.1 运用“五步法”分析 LED 控制逻辑
1. **工作原理**：LED 为共阳极接法，**低电平 (0) 点亮，高电平 (1) 熄灭**。
2. **硬件连接**：查阅开发板的 [📄 野火imx8m底板原理图.pdf](../references/野火imx8m底板原理图.pdf)。
   通过查阅原理图，我们看到了如下的电路连接结构：

```text
彩色LED 原理图结构：

           VDD_3V3 (高电平)
             |
             +----[ D8 RED ]----[ R35 470R ]---- LED_R << GPIO1_IO12
             |
             +----[ D9 GREEN]---[ R36 1.5K ]---- LED_G << GPIO1_IO13
             |
             +----[ D10 BLUE ]--[ R37 470R ]---- LED_B << GPIO1_IO14

说明：三个 LED 阳极共接 3.3V，阴极串联限流电阻后接主控 GPIO。
      只有当 GPIO 输出低电平 (0) 时，电路导通，LED 才会点亮 (Active-low)。
```

3. **寄存器地址**：查阅 i.MX8MM 的 [📄 芯片器件手册 RM](../references/IMX8MMRM.pdf)，查得 GPIO1 控制器基地址为 `0x30200000`。其中方向寄存器 `GDIR` 偏移量为 `0x04`，数据寄存器 `DR` 偏移量为 `0x00`。
4. **配置模式**：将 `GDIR` 的第 12 位置 1，配置为**输出模式**。
5. **执行控制**：将 `DR` 的第 12 位清 0 (`&= ~`) 即可点亮，置 1 (`|=`) 即可熄灭。

#### 1.4.2 编写裸机工程代码
我们将生成包含主 C 代码、极简启动汇编、链接脚本、Makefile 和 Renode 仿真脚本的完整工程。


In [1]:
%%writefile /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/led_baremetal.c
#include <stdint.h>

// 1. 规则 2：使用结构体映射寄存器
typedef struct {
    volatile uint32_t DR;    // 0x00 Data Register
    volatile uint32_t GDIR;  // 0x04 Direction Register
} GPIO_Regs;

// 2. 规则 1：将硬件基地址映射为 volatile 结构体指针
#define GPIO1_BASE 0x30200000
GPIO_Regs *gpio1 = (GPIO_Regs *)GPIO1_BASE;

#define RED_LED_PIN 12

// 简单的软件延时函数（裸机中常见的粗略延时）
void delay(uint32_t count) {
    // 必须加 volatile 防止编译器把空循环优化掉
    for (volatile uint32_t i = 0; i < count; i++) {
        // Do nothing
    }
}

int main(void) {
    // 3. 规则 3：位操作 - 配置 GPIO1_IO12 为输出模式
    gpio1->GDIR |= (1 << RED_LED_PIN);
    
    // 4. 裸机主循环 (Super-loop)
    while (1) {
        // 点亮红灯：因为是 active-low，所以对应位清 0
        gpio1->DR &= ~(1 << RED_LED_PIN);
        delay(500000);
        
        // 熄灭红灯：对应位置 1
        gpio1->DR |= (1 << RED_LED_PIN);
        delay(500000);
    }
    return 0;
}


Writing /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/led_baremetal.c


**补充说明：裸机运行的基础设施（对应 1.1 与 1.1.2）**

在 **1.1.2 节** 我们提到，真实的硬件启动流程是：`上电 -> BootROM -> eMMC 加载 -> 放入 DDR RAM`。在进入我们写的 C 语言 `main` 函数之前，CPU 需要一个运行环境，特别是**栈（Stack）**。因为 C 语言的局部变量和函数调用依赖于栈。下面的 `startup.S` 就是用来搭建这个桥梁的，它位于 RAM 中执行。

在 **1.1 节** 我们学习了 C 程序被编译后，会进行物理切片，分为 `.text` (代码)、`.data` (已初始化数据)、`.bss` (未初始化数据) 等段。在有操作系统的环境中，系统会自动分配这些段的物理地址；但在裸机中，我们需要通过 **链接脚本 (`imx8mm.ld`)** 显式地告诉链接器，将这些段放在物理内存（DDR，即 `0x40000000` 起始的位置）的什么地方。

In [2]:
%%writefile /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/startup.S
// 极简裸机启动汇编：为 C 语言执行准备环境
// 【对应 1.1.2 硬件执行闭环】：CPU 上电后必须先有栈，才能执行 C 代码中的局部变量和函数跳转
.global _start
_start:
    // 初始化栈指针指向内存高地址 (0x40080000, 距离 DDR 起始地址 0x40000000 有 512KB 偏移)
    ldr x30, =0x40080000
    mov sp, x30
    
    // 栈准备好后，跳转到 main，正式进入 C 语言世界
    bl main

    // 如果 main 返回，死循环停机 (裸机程序通常是一个死循环，不应返回)
1:  b 1b


Writing /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/startup.S


In [3]:
%%writefile /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/imx8mm.ld
/* 极简链接脚本：指导编译器如何放置机器码到物理内存 */
ENTRY(_start)
SECTIONS {
    /* 【对应 1.1.2 上电加载】：将程序的入口物理地址定在 0x40000000 (i.MX8MM 的外部 DDR 内存起始物理地址) */
    /* 模拟 BootROM 从 eMMC 将程序搬运到 DDR 的这个位置 */
    . = 0x40000000;
    
    /* 【对应 1.1 编译与物理切片】：将编译产物中的各个段，依次按序排列在 DDR 中 */
    .text : { *(.text) }      /* 存放机器指令 (只读) */
    .data : { *(.data) }      /* 存放已初始化的全局变量 (可读写) */
    .bss : { *(.bss COMMON) } /* 存放未初始化的全局变量 (默认清零) */
}


Writing /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/imx8mm.ld


💡 **温馨提示**：Makefile文件写法可参考**教材** P90-97 3.3 Make工具的使用

In [4]:
%%writefile /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/Makefile
CC = aarch64-linux-gnu-gcc
CFLAGS = -nostdlib -ffreestanding -O0 -g
LDFLAGS = -T imx8mm.ld

all:
	$(CC) $(CFLAGS) $(LDFLAGS) startup.S led_baremetal.c -o led_baremetal.elf

clean:
	rm -f *.elf


Writing /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/Makefile


#### 1.4.3 使用 Renode 模拟硬件并验证 MMIO
Renode 的强大之处在于可以通过简单的文本定义出一个硬件环境。以下是针对本实验的 `.repl` 平台文件和 `.resc` 运行脚本。


**1. 硬件平台描述文件 (`.repl`)**

`.repl` (REnode PLatform) 文件用于定义仿真环境中的虚拟硬件拓扑结构。在下面的脚本中：
- 我们定义了中断控制器 (`GIC`) 并绑定到系统总线。
- 实例化了一颗 `Cortex-A53` (ARMv8A) 核心。
- 在物理地址 `0x40000000` 映射了 128MB 的 DDR 内存。
- 在物理地址 `0x30200000` 映射了 64KB 的 GPIO1 寄存器空间。

*注：为避免无头终端环境下的中文解析乱码，文件内部采用纯英文注释。*

In [5]:
%%writefile /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/imx8mm.repl
// GIC (Generic Interrupt Controller) definition for ARMv8
gic: IRQControllers.ARM_GenericInterruptController @ {
    sysbus new Bus.BusMultiRegistration { address: 0x38800000; size: 0x10000; region: "distributor" };
    sysbus new Bus.BusMultiRegistration { address: 0x38880000; size: 0x10000; region: "cpuInterface" }
}
    0 -> cpu@0
    architectureVersion: IRQControllers.ARM_GenericInterruptControllerVersion.GICv3

// CPU definition: Cortex-A53 (ARMv8A)
cpu: CPU.ARMv8A @ sysbus
    cpuType: "cortex-a53"
    genericInterruptController: gic

// Main Memory (DDR) mapped at 0x40000000, size 128MB (0x08000000)
ram: Memory.MappedMemory @ sysbus 0x40000000
    size: 0x08000000

// GPIO1 Controller mapped at 0x30200000, size 64KB (0x10000)
gpio1: Memory.MappedMemory @ sysbus 0x30200000
    size: 0x10000


Writing /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/imx8mm.repl


**2. 仿真运行控制脚本 (`.resc`)**

`.resc` (REnode SCript) 文件用于控制仿真器的运行流程。在下面的脚本中：
- `mach create`: 创建一个新的机器上下文。
- `machine LoadPlatformDescription`: 加载我们刚才定义的 `.repl` 硬件平台。
- `sysbus LoadELF`: 将交叉编译出来的 `.elf` 可执行文件加载到虚拟内存中。
- `emulation RunFor`: 控制虚拟 CPU 运行指定的时间，并在中途暂停以读取 GPIO 寄存器的值，从而验证 LED 的亮灭逻辑。

In [6]:
%%writefile /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/run.resc
# Create a new machine context
mach create

# Load the platform definition (.repl file)
machine LoadPlatformDescription @/home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/imx8mm.repl

# Load the compiled ELF executable into virtual RAM
sysbus LoadELF @/home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/led_baremetal.elf

# Let the program run for 0.1 virtual seconds to finish GPIO initialization
emulation RunFor "0.1"
echo "====== Check GDIR (Direction) Register ======"
sysbus ReadDoubleWord 0x30200004

# Run for 1.0 virtual second to hit the first LED state
emulation RunFor "1.0"
echo "====== Check DR (Data) Register - 1st Time (High, LED OFF) ======"
sysbus ReadDoubleWord 0x30200000

# Run for 2.0 virtual seconds to pass the C delay loop and hit the second state
emulation RunFor "2.0"
echo "====== Check DR (Data) Register - 2nd Time (Low, LED ON) ======"
sysbus ReadDoubleWord 0x30200000

# Run for 2.0 virtual seconds to pass the loop again and hit the third state
emulation RunFor "2.0"
echo "====== Check DR (Data) Register - 3rd Time (High, LED OFF) ======"
sysbus ReadDoubleWord 0x30200000


Writing /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode/run.resc


In [ ]:
# 1. 编译裸机程序
!cd /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode && make all

# 2. 使用 Renode (如已安装) 运行脚本查看仿真结果。由于是无头环境，我们使用控制台模式运行。
!cd /home/frank/embedded/examples/ch03_imx8m_overview/ch1_baremetal_renode && /home/frank/renode_portable/renode --console --disable-gui run.resc


---

## 附录：在 Ubuntu 22.04 中安装与使用 Renode (Portable版)

在前面的实验中，我们使用了 Renode 来仿真 i.MX8MM 裸机环境。由于国内网络环境限制，源码编译或使用包管理器下载可能会遇到网络连接失败的问题。最稳定、快捷的方式是使用官方预编译的 **Linux Portable** 版本。

### 1. 下载预编译压缩包
建议直接在 Ubuntu 的浏览器中，或者宿主机的浏览器中下载以下文件，然后将其放入 Ubuntu 系统内（例如放在 `~/embedded/` 目录下）：
[👉 点击下载 Renode 1.15.3 Linux Portable](https://github.com/renode/renode/releases/download/v1.15.3/renode-1.15.3.linux-portable.tar.gz)

### 2. 解压与环境配置
打开 Ubuntu 终端 (Terminal)，执行以下命令进行解压和环境变量配置：

```bash
# 1. 创建目标文件夹
mkdir -p ~/renode_portable

# 2. 将压缩包解压到目标文件夹 (请确保当前目录下有刚刚下载的压缩包)
tar xf renode-1.15.3.linux-portable.tar.gz -C ~/renode_portable --strip-components=1

# 3. 将 renode 命令添加到当前用户的环境变量中
echo 'export PATH="/home/frank/renode_portable:$PATH"' >> ~/.bashrc

# 4. 重新加载配置使其生效
source ~/.bashrc

# 5. 验证安装是否成功
renode --version
```

如果成功输出版本号（如 `Renode, version 1.15.3...`），则表示安装成功。

### 3. 在 Jupyter Notebook 中的注意事项
由于 Jupyter Notebook 后台服务可能无法实时读取到新写入 `~/.bashrc` 的环境变量，在 Notebook 中调用 Renode 时，建议使用**绝对路径**以保证执行的稳定性。例如：

```bash
!/home/frank/renode_portable/renode --disable-gui --console run.resc
```

### 4. Renode 核心脚本编写指南 (.repl 与 .resc)

在 Renode 中进行硬件仿真时，有两个文件是必不可少的：`.repl` 和 `.resc`。

#### 4.1 硬件平台描述文件 (`.repl` - REnode PLatform)
**作用**：用来告诉仿真器“你的主板上都有什么零件，它们是怎么连线的”。这相当于硬件工程师画的**原理图**。
**核心概念：`sysbus` (System Bus)**
- 在真实的 ARM 芯片内部，CPU 并不是直接连着内存和 GPIO 的，它们都挂载在一根叫“系统总线（System Bus，如 AXI/AHB）”的通讯主干道上。
- 在 `.repl` 文件中，`sysbus` 就是这根虚拟的系统总线。所有的外设、内存都必须使用 `@ sysbus <物理地址>` 的语法“挂载”到总线上。

**编写示例：**
```text
// 1. 声明一个 128MB 的物理内存，挂载在系统总线的 0x40000000 地址上
ram: Memory.MappedMemory @ sysbus 0x40000000
    size: 0x08000000

// 2. 声明一个 Cortex-A53 的 CPU，也挂载在总线上
cpu: CPU.ARMv8A @ sysbus
    cpuType: "cortex-a53"
```

#### 4.2 仿真运行控制脚本 (`.resc` - REnode SCript)
**作用**：用来告诉仿真器“启动后该做什么动作”。这相当于你在做物理实验时的**实验步骤指导书**。

**常用命令：**
- `mach create`：创建一台全新的虚拟电脑（Machine Context）。
- `machine LoadPlatformDescription @xxx.repl`：把刚才写的“原理图”加载进这台虚拟电脑里。
- `sysbus LoadELF @xxx.elf`：相当于我们用 J-Link 或 eMMC 把编译好的程序烧录到内存里。注意，这里也是通过 `sysbus` (系统总线) 往内存地址里写数据的。
- `sysbus ReadDoubleWord 0x30200000`：通过系统总线去读取 GPIO 寄存器的值（相当于用万用表测引脚电平）。
- `emulation RunFor "1.0"`：让虚拟时间精准往前跑 1.0 秒。

**编写示例：**
```text
mach create
machine LoadPlatformDescription @imx8mm.repl
sysbus LoadELF @led_baremetal.elf
emulation RunFor "1.0"
sysbus ReadDoubleWord 0x30200000
```
